# Test PTA-simulator

We simulate a small PTA with i.i.d. zero mean Gaussian white noise, intrinsic pulsar noise, and a stochastic GWB with PTA-simulator. We then perform parameter estimation with Prometheus.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import corner

from pulsar import Pulsar, simulate_toas
from pta import PTA
from data import SimulatedData

from prometheus.spectral_models import CommonSpectralModel, IndependentSpectralModel
from prometheus import spectra
from prometheus.pta_model import PTAModel
import numpyro
import numpyro.distributions as dist
import jax.numpy as jnp
import jax.random as jr

libstempo not installed. PINT or libstempo are required to use par and tim files.


## Simulate data

In [2]:
# create simulated pulsars
npsrs = 20
psrs = []
for i in range(npsrs):
    toas = simulate_toas(53_000, 15., seed=i)
    psr = Pulsar(name=f'PSR{i}',
                 costheta=np.random.uniform(-1, 1),
                 phi=np.random.uniform(0, 2 * np.pi),
                 dist_kpc=np.random.uniform(0.5, 5.),
                 dist_kpc_std=0.2,
                 toas=toas,
                 toa_errors=np.ones_like(toas) * 0.5e-6)
    psrs.append(psr)

In [3]:
# create PTA
pta = PTA(psrs)

# add white noise
pta.add_white_noise(seed=234)

# add intrinsic pulsar noise and a GWB
log10_A_gwb = -14.5
gamma_gwb = 13 / 3
log10_As_rn = np.random.uniform(-18, -14, size=npsrs)
gammas_rn = np.random.uniform(2, 5, size=npsrs)
pta.add_irn_gwb_delay(log10_As_rn, gammas_rn, log10_A_gwb, gamma_gwb, seed=345)

In [4]:
# create prometheus compatible data object
data = SimulatedData(pta, name='simulated_data')

building pulsar models: 100%|██████████| 20/20 [00:00<00:00, 283.88it/s, running PSR19]


## Parameter estimation

In [5]:
# spectral models
psr_model = IndependentSpectralModel(name='psr_params',
                                     parameter_bounds=[[-20., -10.],
                                                       [0., 7.]],
                                     data=data,
                                     get_phi_diag_func=spectra.power_law)
gwb_model = CommonSpectralModel(name='gwb_params',
                                parameter_bounds=[[-20., -10.],
                                                [0., 7.]],
                                data=data,
                                get_phi_diag_func=spectra.power_law,
                                correlation_matrix='HD')
pta_model = PTAModel(psr_model=psr_model,
                     gwb_model=gwb_model)

In [8]:
# test posterior evaluation
psr_params = jnp.array([[-15., 4.]] * npsrs)
gwb_params = jnp.array([-14.5, 13 / 3])
z = jnp.zeros((pta_model.npsrs, pta_model.ncomponents))
print(pta_model.ln_posterior(z, psr_params, gwb_params))

28209.66


In [10]:
%timeit pta_model.ln_posterior(z, psr_params, gwb_params)

246 μs ± 6.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
